# Quiz (locked)
The quiz is encrypted. When the quiz starts, type the password into the cell below (between the quotes) and run it. It creates the quiz notebook in this folder.

In [ ]:
PASSWORD = ""  # <-- type the password, then run this cell

import base64
import hashlib
import json


def _unlock(password, box):
    h = bytes.fromhex(box["salt"]) + password.encode()
    for _ in range(box["iter"]):
        h = hashlib.sha256(h + password.encode()).digest()
    key, nonce, ct = h, bytes.fromhex(box["nonce"]), base64.b64decode(box["data"])
    if hashlib.sha256(key + b"tag" + ct).hexdigest() != box["tag"]:
        raise ValueError("Wrong password - please check it and run the cell again.")
    out = bytearray()
    for i in range(0, len(ct), 32):
        block = hashlib.sha256(key + nonce + i.to_bytes(8, "big")).digest()
        out += bytes(a ^ b for a, b in zip(ct[i : i + 32], block))
    for name, content in json.loads(out).items():
        with open(name, "wb") as f:
            f.write(base64.b64decode(content))
        print("created", name)
    print("Done. Now you can open the quiz notebook.")


with open("quiz_locked.json") as f:
    _unlock(PASSWORD, json.load(f))